# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [ ]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [12]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [13]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [14]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [15]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [16]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [18]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [19]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which is mentioned multiple times in the sample. However, since only a subset of the data is shown and the total counts are not explicitly provided, I cannot definitively confirm the overall most common domain. \n\nIf considering only the sample, "Healthcare / MedTech" and "Creative / Design / Media" both appear multiple times, but "Healthcare / MedTech" is slightly more frequent in the visible entries. \n\nFor an accurate answer covering the entire dataset, a complete count across all entries would be necessary. \n\nWould you like me to analyze the full dataset to determine the most common project domain?'

In [20]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, one project titled "LatticeFlow" in the Healthcare / MedTech domain with a secondary focus on Security involves an AI-powered platform optimizing logistics routes for sustainability.'

In [21]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects were generally positive. They described some projects as clever solutions with measurable environmental benefits, comprehensive and technically mature approaches, promising ideas supported by robust validation, and solid work with impressive real-world impact. For example, one project was praised as a "clever solution with measurable environmental benefit," another as "comprehensive and technically mature," and others noted "impressive real-world impact" and "excellent code quality." Overall, the judges appreciated the technical quality, innovation, and potential impact of the fintech-related projects.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [27]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [28]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [24]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is not explicitly stated as the most frequent in the sample. The sample shows projects in the domains of "Productivity Assistants," "Legal / Compliance," "Data / Analytics," and "Healthcare / MedTech." Without a larger dataset, I cannot determine which domain is most common overall. Therefore, I don\'t know the most common project domain.'

In [29]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any usecases specifically related to security.'

In [30]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judge comments about the fintech projects indicate that the projects were viewed positively in terms of technical execution and ambition. Specifically, the project "PulseAI 50" in the Finance / FinTech domain was described as "Technically ambitious and well-executed."'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer ✅ 

BM25 excels for queries with rare, specific terms like acronyms ("HIPAA compliance"), product codes ("SKU-12345"), or technical identifiers because it matches exact keywords rather than semantic similarity, ensuring precise retrieval when the exact term must be present.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [31]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [32]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [33]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned in at least one of the entries. However, since the sample is limited, I cannot conclusively determine the most common domain across all projects. If more information or additional data were available, I could provide a more accurate answer.'

In [34]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security. The projects mentioned focus on privacy improvements in healthcare applications through federated learning, but there is no direct mention of security use cases.'

In [35]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments regarding the fintech projects. Specifically, for the "PlanPilot" project in the Finance/FinTech domain, the judges described it as "a clever solution with measurable environmental benefit" and gave it a high score of 8.4.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [36]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [37]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [38]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Healthcare / MedTech," which is mentioned multiple times across different projects.'

In [39]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there were use cases related to security. One example is the project titled "SecureNest 49," which involves a document summarization and retrieval system for enterprise knowledge bases, addressing legal and compliance security needs.'

In [40]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had generally positive comments about the fintech projects. For example, they described the "SecureNest" project as having a "conceptually strong" approach, noting that it needed more benchmarking results. The "Pathfinder 27" project received praise for "excellent code quality and use of open-source libraries." Overall, the judges highlighted strengths such as strong conceptual ideas, strong execution, scalability, and potential for commercialization. However, some projects were also noted to require further benchmarking or additional results to fully validate their impact.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

Generating multiple reformulations of a user query improves **recall** (the ability to retrieve all relevant documents) through several key mechanisms:

1. **Overcomes Vocabulary Mismatch**: User queries and documents often use different terminology for the same concept. For example, a query about "patient care" might miss documents using "healthcare applications," "medical assistance," or "clinical support." Multiple reformulations with varied vocabulary increase the chance of matching documents regardless of terminology differences.

2. **Captures Different Semantic Angles**: A single query vector represents one semantic perspective, but multiple reformulations explore different semantic spaces of the same information need. For instance, "security usecases" could be reformulated as "cybersecurity solutions," "data protection applications," or "security-focused implementations"—each capturing different aspects.

3. **Increases Coverage Through Set Union**: Each reformulated query retrieves its own set of documents, and the Multi-Query Retriever takes the **union** of all unique documents. This expands the total pool of retrieved documents, significantly increasing the probability that all relevant documents are included.

4. **Compensates for Embedding Limitations**: Single embeddings may not capture all nuances of complex or ambiguous queries. Distributing the semantic load across multiple query embeddings (typically 3-5 variations) provides more comprehensive coverage of the semantic space.

**Trade-offs**: While multi-query retrieval improves recall, it comes at the cost of higher latency (multiple retrieval operations), increased expenses (additional LLM and embedding API calls), and potentially lower precision (more irrelevant documents may be retrieved alongside relevant ones).


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [41]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [42]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [43]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [44]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [45]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [46]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided data, the most common project domain is not explicitly stated, but considering the sample projects listed, the domains mentioned are Security, Healthcare / MedTech, Productivity Assistants, and Creative / Design / Media. Since only a few entries are shown, I cannot definitively determine the most common domain overall. However, if we consider the sample, the Domain 'Healthcare / MedTech' appears more than once (for example, in the projects BioForge and SkyForge). \n\nTherefore, based on this sample, Healthcare / MedTech seems to be the most common project domain."

In [47]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit mentions of use cases specifically related to security. The projects listed primarily focus on privacy improvements in healthcare applications and related domains, but do not specify security use cases.'

In [48]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech-related project "SkyForge." They described it as a "clever solution with measurable environmental benefit" and gave it a high score of 94, indicating a strong appreciation for its innovation and impact.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [49]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [50]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [51]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Legal / Compliance," which appears multiple times.'

In [52]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there is at least one use case related to security. The project titled "SecureNest" in the E‑commerce / Marketplaces domain involves a document summarization and retrieval system for enterprise knowledge bases, which can enhance security by improving information management and access control.'

In [53]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments about the fintech projects were generally positive. Specifically, for the project "Pathfinder," which falls under the finance/fintech domain, the judges described it as having "Excellent code quality and use of open-source libraries." This indicates a high level of technical proficiency and effective use of available resources.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [54]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [55]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [56]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [57]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [58]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [59]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Legal / Compliance," which appears twice among the listed projects.'

In [60]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the projects "SynthMind" and "BioForge" are associated with the security domain.'

In [61]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had varied comments about the fintech projects. One project, "TrendLens 19," was described as "Technically ambitious and well-executed," with a judge score of 8.9. Another project, "AutoMate 5," was noted for being "A forward-looking idea with solid supporting data," receiving a judge score of 8.6. Overall, judges appreciated the technical ambition and solid execution of the fintech-related projects, highlighting their innovative potential and strong implementation.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

With short, highly repetitive sentences like FAQs, semantic chunking can exhibit problematic behaviors:

**Expected Behavior Problems:**

1. **Narrow Similarity Distribution**: FAQ sentences often use similar language patterns and structure (e.g., "How do I...", "What is...", "Can I..."), resulting in uniformly high semantic similarity scores with little variance.

2. **Overly Large Chunks**: With percentile-based thresholds, if most sentences are semantically similar, the algorithm may group too many FAQs together, creating chunks that lose granularity and making it harder to retrieve specific Q&A pairs.

3. **Overly Small Chunks**: Alternatively, subtle word differences might trigger splits at inappropriate boundaries, creating overly fragmented chunks where each FAQ question becomes its own document, missing the benefit of contextual grouping.

4. **Loss of Q&A Pairing**: Semantic chunking might split questions from their answers if they're semantically different, breaking the logical structure of FAQ content.

**Algorithm Adjustments:**

1. **Change Breakpoint Threshold Type**: Switch from `percentile` to `gradient` or `interquartile` methods, which are better at detecting subtle shifts in semantic similarity patterns when the overall distribution is narrow.

2. **Hybrid Structural + Semantic Approach**: Combine semantic chunking with rule-based logic:
   - Detect Q&A pairs using structural markers (question marks, formatting patterns)
   - Keep Q&A pairs together as atomic units
   - Apply semantic chunking to group related Q&A pairs by topic

3. **Adjust Threshold Sensitivity**: Fine-tune breakpoint thresholds to be more sensitive to small semantic differences. For percentile-based chunking, use lower percentiles (e.g., 50th instead of 95th percentile).

4. **Pre-processing with Metadata**: Add categorical metadata to FAQs (topic tags, categories) before chunking, then apply semantic chunking within categories rather than across the entire FAQ corpus.

5. **Consider Alternative Chunking**: For highly structured FAQs, semantic chunking might not be optimal. Consider:
   - Fixed-size chunking (e.g., 3-5 Q&A pairs per chunk)
   - Rule-based chunking based on category headers or section markers
   - Topic modeling to group FAQs by latent topics before semantic chunking


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

### Step 1: Import Ragas and Dependencies

First, let's import the necessary Ragas libraries for evaluation (v0.3.6). Note: If you haven't installed ragas yet, run: `uv add ragas datasets matplotlib`


In [66]:
# Ragas imports for evaluation (v0.3.6)
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)
from datasets import Dataset
import pandas as pd


### Step 2: Create a Golden Test Dataset

We'll create a curated test dataset with questions and ground truth answers relevant to our project data. This manual approach ensures quality and is compatible with Ragas 0.3.6.


In [67]:
# Create a golden dataset manually
# For ragas 0.3.6, we'll create a curated test set based on our data

# Define test questions and ground truth answers
test_questions = [
    "What is the most common project domain?",
    "Were there any usecases about security?",
    "What did judges have to say about the fintech projects?",
    "What projects were in the healthcare domain?",
    "Which projects received the highest scores?",
    "Are there any AI-powered education projects?",
    "What domains appear in the dataset?",
    "Which projects focus on natural language processing?",
    "What are some examples of machine learning projects?",
    "What types of AI applications are represented in the projects?"
]

# Ground truth answers (you can refine these based on your actual data)
test_ground_truths = [
    "Based on the dataset, the most common project domains need to be counted from the available projects.",
    "Yes, there are security-related projects in the dataset.",
    "Judges provided feedback on fintech projects regarding their innovation, feasibility, and impact.",
    "Healthcare projects include AI applications for medical diagnosis, patient care, and health monitoring.",
    "The highest scoring projects are those with scores above 85.",
    "Yes, there are education-focused AI projects in the dataset.",
    "The dataset contains multiple domains including healthcare, fintech, education, and others.",
    "NLP projects involve text analysis, chatbots, and language understanding applications.",
    "Machine learning projects include predictive models, classification systems, and data analysis tools.",
    "The projects represent various AI applications including computer vision, NLP, ML, and data science."
]

# Create a simple testset dictionary
testset_dict = {
    "user_input": test_questions,
    "reference": test_ground_truths
}


In [68]:
# Convert to DataFrame for viewing
test_df = pd.DataFrame(testset_dict)
print(f"Created {len(test_df)} test questions")
test_df.head()


Created 10 test questions


,user_input,reference
0,What is the most common project domain?,"Based on the dataset, the most common project ..."
1,Were there any usecases about security?,"Yes, there are security-related projects in th..."
2,What did judges have to say about the fintech ...,Judges provided feedback on fintech projects r...
3,What projects were in the healthcare domain?,Healthcare projects include AI applications fo...
4,Which projects received the highest scores?,The highest scoring projects are those with sc...


### Step 3: Setup LangSmith for Cost and Latency Tracking

LangSmith will help us track the cost and latency of each retriever method.


In [69]:
# Optional: Configure LangSmith for detailed tracing
# Uncomment and set your LangSmith API key if you want detailed tracking
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key:")
# os.environ["LANGCHAIN_PROJECT"] = "Advanced_Retrieval_Evaluation"

import time


### Step 4: Create Helper Function for Evaluation

We'll create a helper function that evaluates a retrieval chain using our synthetic test set and Ragas metrics.


In [70]:
def evaluate_retrieval_chain(chain, retriever_name, test_questions, test_ground_truths):
    """
    Evaluate a retrieval chain using Ragas metrics.
    
    Args:
        chain: The retrieval chain to evaluate
        retriever_name: Name of the retriever for tracking
        test_questions: List of test questions
        test_ground_truths: List of ground truth answers
    
    Returns:
        Dictionary containing evaluation results
    """
    print(f"\n{'='*50}")
    print(f"Evaluating {retriever_name}")
    print(f"{'='*50}")
    
    # Store responses and contexts
    answers = []
    retrieved_contexts = []
    latencies = []
    
    # Run the chain for each question
    for question in test_questions:
        start_time = time.time()
        try:
            result = chain.invoke({"question": question})
            latency = time.time() - start_time
            
            answers.append(result["response"].content)
            retrieved_contexts.append([doc.page_content for doc in result["context"]])
            latencies.append(latency)
        except Exception as e:
            print(f"Error processing question: {question}")
            print(f"Error: {str(e)}")
            answers.append("")
            retrieved_contexts.append([])
            latencies.append(0)
    
    # Create dataset for Ragas
    data = {
        "question": test_questions,
        "answer": answers,
        "contexts": retrieved_contexts,
        "ground_truth": test_ground_truths
    }
    
    dataset = Dataset.from_dict(data)
    
    # Evaluate with Ragas metrics
    result = evaluate(
        dataset,
        metrics=[
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy,
        ],
    )
    
    # Calculate average latency
    avg_latency = sum(latencies) / len(latencies) if latencies else 0
    
    # Return results
    return {
        "retriever": retriever_name,
        "metrics": result,
        "avg_latency_seconds": avg_latency,
        "total_queries": len(test_questions)
    }


### Step 5: Prepare Test Data


In [72]:
# Extract test data from the testset
test_questions_list = test_df['user_input'].tolist()
test_ground_truths_list = test_df['reference'].tolist()

print(f"Prepared {len(test_questions_list)} test questions for evaluation")


Prepared 10 test questions for evaluation


### Step 6: Evaluate Each Retriever

Now we'll evaluate each retrieval method we've implemented.


In [ ]:
# Store all evaluation results
all_results = []

# Define the retrievers and chains to evaluate
retrievers_to_evaluate = [
    ("Naive Retrieval", naive_retrieval_chain),
    ("BM25 Retrieval", bm25_retrieval_chain),
    ("Contextual Compression (Rerank)", contextual_compression_retrieval_chain),
    ("Multi-Query Retrieval", multi_query_retrieval_chain),
    ("Parent Document Retrieval", parent_document_retrieval_chain),
    ("Ensemble Retrieval", ensemble_retrieval_chain),
]

# Evaluate each retriever
for name, chain in retrievers_to_evaluate:
    result = evaluate_retrieval_chain(
        chain,
        name,
        test_questions_list,
        test_ground_truths_list
    )
    all_results.append(result)


### Step 7: Compile and Compare Results


In [ ]:
# Compile results into a DataFrame for easy comparison
comparison_data = []

for result in all_results:
    metrics_dict = result["metrics"]
    
    comparison_data.append({
        "Retriever": result["retriever"],
        "Context Precision": metrics_dict.get("context_precision", 0),
        "Context Recall": metrics_dict.get("context_recall", 0),
        "Faithfulness": metrics_dict.get("faithfulness", 0),
        "Answer Relevancy": metrics_dict.get("answer_relevancy", 0),
        "Avg Latency (s)": result["avg_latency_seconds"],
    })

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Sort by a composite score (you can adjust the weights)
comparison_df["Overall Score"] = (
    comparison_df["Context Precision"] * 0.25 +
    comparison_df["Context Recall"] * 0.25 +
    comparison_df["Faithfulness"] * 0.25 +
    comparison_df["Answer Relevancy"] * 0.25
)

comparison_df = comparison_df.sort_values("Overall Score", ascending=False)
comparison_df


### Step 8: Visualize Results


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Context Precision
axes[0, 0].barh(comparison_df["Retriever"], comparison_df["Context Precision"], color='skyblue')
axes[0, 0].set_xlabel('Context Precision')
axes[0, 0].set_title('Context Precision by Retriever')
axes[0, 0].set_xlim(0, 1)

# Plot 2: Context Recall
axes[0, 1].barh(comparison_df["Retriever"], comparison_df["Context Recall"], color='lightgreen')
axes[0, 1].set_xlabel('Context Recall')
axes[0, 1].set_title('Context Recall by Retriever')
axes[0, 1].set_xlim(0, 1)

# Plot 3: Overall Score
axes[1, 0].barh(comparison_df["Retriever"], comparison_df["Overall Score"], color='coral')
axes[1, 0].set_xlabel('Overall Score')
axes[1, 0].set_title('Overall Score by Retriever')
axes[1, 0].set_xlim(0, 1)

# Plot 4: Latency
axes[1, 1].barh(comparison_df["Retriever"], comparison_df["Avg Latency (s)"], color='plum')
axes[1, 1].set_xlabel('Average Latency (seconds)')
axes[1, 1].set_title('Average Latency by Retriever')

plt.tight_layout()
plt.show()


### Step 9: Cost Analysis

Let's analyze the cost implications of each retriever. Cost considerations include:

1. **Embedding Costs**: Some retrievers require more embedding operations
2. **LLM Costs**: Multi-query retriever generates additional queries, reranking uses additional API calls
3. **Storage**: Vector stores and document stores have storage costs

For detailed cost tracking, use LangSmith which provides token usage and cost per run.


In [ ]:
# Cost considerations for each retriever
cost_analysis = {
    "Naive Retrieval": {
        "embedding_calls": "One per query",
        "llm_calls": "One per query",
        "additional_api_costs": "None",
        "relative_cost": "Low (baseline)"
    },
    "BM25 Retrieval": {
        "embedding_calls": "None (uses term frequency)",
        "llm_calls": "One per query",
        "additional_api_costs": "None",
        "relative_cost": "Lowest (no embeddings)"
    },
    "Contextual Compression (Rerank)": {
        "embedding_calls": "One per query",
        "llm_calls": "One per query",
        "additional_api_costs": "Cohere Rerank API calls",
        "relative_cost": "Medium-High (+ rerank costs)"
    },
    "Multi-Query Retrieval": {
        "embedding_calls": "Multiple per query (3-5x)",
        "llm_calls": "2x per query (query generation + response)",
        "additional_api_costs": "Extra LLM calls for query generation",
        "relative_cost": "High (multiple queries)"
    },
    "Parent Document Retrieval": {
        "embedding_calls": "One per query (but more embeddings stored)",
        "llm_calls": "One per query",
        "additional_api_costs": "Higher storage costs",
        "relative_cost": "Medium (storage overhead)"
    },
    "Ensemble Retrieval": {
        "embedding_calls": "Varies (combines multiple retrievers)",
        "llm_calls": "Multiple (sum of all retrievers)",
        "additional_api_costs": "Sum of all retriever costs",
        "relative_cost": "Highest (all methods combined)"
    }
}

cost_df = pd.DataFrame(cost_analysis).T
cost_df


### Step 10: Final Analysis and Recommendations

Based on the evaluation results above, write your analysis considering:

1. **Performance Metrics**: Which retriever scored highest on Ragas metrics?
2. **Latency**: Which retriever is fastest? Is the speed difference significant?
3. **Cost**: What are the cost implications of each approach?
4. **Use Case Fit**: Which retriever is best suited for this particular dataset (structured project descriptions)?

#### ✅ Your Analysis:

Write your analysis here. Consider the following questions:

- Which retriever method performed best overall?
- What trade-offs exist between performance, cost, and latency?
- For this specific use case (project descriptions with structured metadata), which method would you recommend and why?
- Are there scenarios where you might choose a different retriever?

**Example structure:**

Based on the evaluation results:

1. **Best Overall Performer**: [Retriever name] achieved the highest overall score of [X], with particularly strong performance in [metric].

2. **Cost-Effectiveness**: [Retriever name] offers the best balance of performance and cost because [reason].

3. **Latency Considerations**: [Retriever name] has the lowest latency at [X] seconds, making it ideal for real-time applications.

4. **Recommendation**: For this dataset of structured project descriptions, I recommend [retriever name] because:
   - [Reason 1 related to data characteristics]
   - [Reason 2 related to performance metrics]
   - [Reason 3 related to cost/latency trade-offs]

5. **Alternative Scenarios**: 
   - If cost is the primary concern: [recommendation]
   - If latency is the primary concern: [recommendation]
   - If maximum accuracy is required regardless of cost: [recommendation]


#### Write Your Analysis Here

[Add your analysis after running the evaluations]


### Bonus: Evaluate Semantic Chunking (Optional)

While semantic chunking is a preprocessing technique rather than a retrieval method, you can compare its impact by evaluating the semantic retrieval chain.


In [ ]:
# Evaluate semantic chunking (optional)
semantic_result = evaluate_retrieval_chain(
    semantic_retrieval_chain,
    "Semantic Chunking Retrieval",
    test_questions_list,
    test_ground_truths_list
)

# Add to results
print("\nSemantic Chunking Results:")
print(f"Metrics: {semantic_result['metrics']}")
print(f"Avg Latency: {semantic_result['avg_latency_seconds']:.3f} seconds")


### 📋 Summary and Next Steps

**What You've Accomplished:**
1. ✅ Created a synthetic golden dataset using Ragas
2. ✅ Evaluated 6 different retrieval methods with Ragas metrics
3. ✅ Analyzed performance, latency, and cost trade-offs
4. ✅ Created visualizations to compare retrievers

**Key Metrics Explained:**
- **Context Precision**: How relevant are the retrieved contexts to the question?
- **Context Recall**: How much of the necessary information was retrieved?
- **Faithfulness**: Is the answer grounded in the retrieved context?
- **Answer Relevancy**: How relevant is the answer to the question?

**Tips for Analysis:**
- Look for retrievers that balance all metrics, not just one
- Consider your specific use case requirements (cost vs. performance vs. latency)
- Think about scalability: some methods work better at larger scales
- Remember that ensemble methods combine strengths but increase costs

**For LangSmith Integration:**
- Uncomment the LangSmith configuration in Step 3
- View detailed traces at https://smith.langchain.com
- Analyze token usage and exact costs per retriever
- Compare latency distributions across multiple runs
